In [22]:
import pickle

with open("../data/preprocessed_data.pkl", "rb") as f:
    preprocessed_data = pickle.load(f)

X_train_scaled = preprocessed_data["X_train"]
X_val_scaled = preprocessed_data["X_val"]
X_test_scaled = preprocessed_data["X_test"]
y_train = preprocessed_data["y_train"]
y_val = preprocessed_data["y_val"]
y_test = preprocessed_data["y_test"]

print(X_train_scaled.shape)
print(X_val_scaled.shape)
print(X_test_scaled.shape)

(1096, 100)
(235, 100)
(236, 100)


In [25]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# 1. use only normal samples for training
X_train_normal = X_train_scaled[y_train == -1]

print("Isolation Forest")
print("Normal training samples shape:", X_train_normal.shape)

# 2. train Isolation Forest
iso = IsolationForest(
    n_estimators=200,
    contamination=0.07,
    random_state=42
)

iso.fit(X_train_normal)

# 3. predict on validation set
val_pred_raw = iso.predict(X_val_scaled)

# convert output:
# IsolationForest:  1 = normal, -1 = anomaly
# Our labels:      -1 = pass,   1 = fail
y_val_pred = [1 if p == -1 else -1 for p in val_pred_raw]

# 4. evaluation
print("Validation Results (Isolation Forest)")
print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("Precision:", precision_score(y_val, y_val_pred, pos_label=1))
print("Recall:", recall_score(y_val, y_val_pred, pos_label=1))
print("F1-score:", f1_score(y_val, y_val_pred, pos_label=1))
print("Confusion Matrix:")
print(confusion_matrix(y_val, y_val_pred, labels=[-1, 1]))

Isolation Forest
Normal training samples shape: (1023, 100)
Validation Results (Isolation Forest)
Accuracy: 0.8680851063829788
Precision: 0.13636363636363635
Recall: 0.2
F1-score: 0.16216216216216217
Confusion Matrix:
[[201  19]
 [ 12   3]]


In [26]:
from sklearn.svm import OneClassSVM
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# use only normal samples for training
X_train_normal = X_train_scaled[y_train == -1]

print("Normal training samples shape:", X_train_normal.shape)

# train One-Class SVM
ocsvm = OneClassSVM(
    kernel="rbf",
    nu=0.05,
    gamma="scale"
)

ocsvm.fit(X_train_normal)

# predict on validation set
val_pred_raw = ocsvm.predict(X_val_scaled)

# One-Class SVM output:
#  1  = normal
# -1  = anomaly
# convert to your label format:
# anomaly -> fail (1)
# normal  -> pass (-1)
y_val_pred = [1 if p == -1 else -1 for p in val_pred_raw]

print("Validation Results (OneClassSVM)")
print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("Precision:", precision_score(y_val, y_val_pred, pos_label=1))
print("Recall:", recall_score(y_val, y_val_pred, pos_label=1))
print("F1-score:", f1_score(y_val, y_val_pred, pos_label=1))
print("Confusion Matrix:")
print(confusion_matrix(y_val, y_val_pred, labels=[-1, 1]))

Normal training samples shape: (1023, 100)
Validation Results (OneClassSVM)
Accuracy: 0.8595744680851064
Precision: 0.09090909090909091
Recall: 0.13333333333333333
F1-score: 0.10810810810810811
Confusion Matrix:
[[200  20]
 [ 13   2]]
